In [ ]:
from pathlib import Path
import sys

# Locate new_massloss whether Jupyter starts here, in joe's_Code,
# or at the repository root. Shared code and potentials live in ngc6569.
_project_candidates = (
    Path.cwd(),
    Path.cwd() / "new_massloss",
    Path.cwd() / "joe's_Code" / "new_massloss",
)
PROJECT_DIR = next(
    (path.resolve() for path in _project_candidates if (path / "MassLossRun 1.ipynb").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not locate the new_massloss project directory")
NGC6569_DIR = PROJECT_DIR.parent / "ngc6569"
MILKYWAY_DIR = NGC6569_DIR / "milkyway"
OUTPUT_DIR = PROJECT_DIR / "outputs"
if not MILKYWAY_DIR.is_dir():
    raise FileNotFoundError(f"Missing shared potential directory: {MILKYWAY_DIR}")
sys.path.insert(0, str(PROJECT_DIR))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 200  # Set to 100MB or whatever you need

from IPython.display import HTML


In [ ]:
import astropy.coordinates as coord
import astropy.units as u
import gala.dynamics as gd

import agama

from time import time

from bound_by_energy import get_bound_by_energy
from cluster_frame import get_cluster_data
from leap_frog import kdk_leapfrog, kdk_leapfrog_TD

In [ ]:
# default Astropy Galactocentric frame parameters to the values adopted in Astropy v4.0:
_ = coord.galactocentric_frame_defaults.set('v4.0')

# set Agama units 
# working units: 1 Msun, 1 kpc, 1 km/s
agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
print("Newton G in Agama units,",agama.G)

# Check the current unit system
print("Current Agama units:")
print(f"Length unit: {agama.getUnits()['length']}")
print(f"Velocity unit: {agama.getUnits()['velocity']}")  
print(f"Time unit: {agama.getUnits()['time']}")
print(f"Mass unit: {agama.getUnits()['mass']}")

agama_time_unit = agama.getUnits()["time"]
print(agama_time_unit)

In [ ]:
# these are three potentials you can choose from
# Note, if you use the rotating potential, then you need to call 
# the time dependent leap-frog algorithm 

# Use the Hunter  potential 
pot_ext = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_full.ini"))
pot_rot = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_rotating.ini"))
pot_bovy = agama.Potential(str(MILKYWAY_DIR / "MWPotential2014.ini"))

pot_use = pot_ext

In [ ]:
# NG6569 coordinates 

c = coord.SkyCoord(ra = 273.412*u.degree, dec = -31.827*u.degree,
                        distance=(10.5)*u.kpc,
                        pm_ra_cosdec= -4.125*u.mas/u.yr,
                        pm_dec= -7.354*u.mas/u.yr,
                        radial_velocity= -49.82*u.km/u.s)

# transform to galactic centeric 
c_gc = c.transform_to(coord.Galactocentric).data
print(c_gc._differentials)

In [ ]:
# creat phase space object
w0 = gd.PhaseSpacePosition(c_gc)
print("initial position :", w0.pos)
print("initial velocity :", w0.vel)
print("Need to convert velocity to km/s")

pos_0 = np.r_[w0.pos.x.value, w0.pos.y.value, w0.pos.z.value]
vel_0 = np.r_[w0.vel.d_x.to(u.km/u.s).value, 
              w0.vel.d_y.to(u.km/u.s).value,
              w0.vel.d_z.to(u.km/u.s).value]
print("position", pos_0) 
print("velocity", vel_0) 

In [ ]:
# check against 
# -31.81767578935911 km / s -174.361128405216 km / s 23.931083813686854 km / s

In [ ]:
# integrate orbit 
tfin= -200*u.Myr
nt=2000
t_eval = np.linspace(0, tfin, nt)
t_scipy = t_eval.to(u.Gyr).value/agama_time_unit.to(u.Gyr).value
t_scipy

In [ ]:
def rhs(t,state): 
    
    pos = state[:3]
    vel = state[3:]

    acc = pot_use.force(pos, t=t)
    return np.r_[vel, acc] 

# integrate with solve_ivp
from scipy.integrate import solve_ivp

state_0 = np.r_[pos_0, vel_0] 
print(state_0, state_0.shape)

t_span = (0, t_scipy[-1]) 
sol = solve_ivp(rhs, t_span, state_0, t_eval =t_scipy, rtol=1e-10, atol=1e-10)
orbit = sol["y"]

In [ ]:
x = orbit[0,:]
y = orbit[1,:]
z = orbit[2,:]
vx = orbit[3,:]
vy = orbit[4,:]
vz = orbit[5,:]

In [ ]:
# Agama orbit
# Calculate orbit
orbit = agama.orbit(potential=pot_use, 
                   ic=state_0, 
                   time=t_span[1],      # Total integration time
                   trajsize=nt)   # Number of output points

In [ ]:
orbit[1].shape

In [ ]:
# 2-D orbit figures 
fig = plt.figure(figsize=(14, 4))

ax = fig.add_subplot(131)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("y [kpc]", size=20)
ax.plot(x,y, label="from scratch")
ax.plot(orbit[1][:,0], orbit[1][:,1], "--", label="Agama") 
ax.legend()

ax = fig.add_subplot(132)
ax.set_xlabel("x [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(x,z)
ax.plot(orbit[1][:,0], orbit[1][:,2], "--") 

ax = fig.add_subplot(133)
ax.set_xlabel("y [kpc]", size=20) 
ax.set_ylabel("z [kpc]", size=20)
ax.plot(y,z)
ax.plot(orbit[1][:,1], orbit[1][:,2], "--") 
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "ngc6569_orbit.pdf")

In [ ]:
# Create figure
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
    
# Create scatter plot with color mapping by radius
ax.plot(x,y,z, alpha = .5) 
ax.set_xlabel('X (pc)')
ax.set_ylabel('Y (pc)')
ax.set_zlabel('Z (pc)')
fig.savefig(OUTPUT_DIR / "ngc6569_3d_orbit.pdf")


In [ ]:
# Define the parameters for your King model
W0_value = 7.0  # Example W0 value
# create an isolated star cluster
r_scale = 1/1000
m = 2.3*1e5*(2)
pot_sat = agama.Potential(type='king', W0=W0_value, scaleRadius=r_scale, mass=m)
df_sat = agama.DistributionFunction(type='quasispherical', potential=pot_sat)
Nbody = 150000
xv, mass = agama.GalaxyModel(pot_sat, df_sat).sample(Nbody)

r_agama = np.sqrt(xv[:,0]**2 + xv[:,1]**2 + xv[:,2]**2) 
v_agama = np.sqrt(xv[:,3]**2 + xv[:,4]**2 + xv[:,5]**2) 

print("Agama G:", agama.G)

cluster_initial_data = np.c_[mass, xv]
cluster_initial_data.shape
np.savetxt(OUTPUT_DIR / "cluster_data.txt", cluster_initial_data)

In [ ]:

# look at positions in 3D
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
    
# Create scatter plot
ax.scatter(xv[:,0], xv[:,1], xv[:,2], alpha=0.1, s=20)
    
# Labels and title
ax.set_xlabel('X [kpc]', fontsize=12)
ax.set_ylabel('Y [kpc]', fontsize=12)
ax.set_zlabel('Z [kpc]', fontsize=12)
    

In [ ]:
fig = plt.figure(figsize=(14,6))

ax=fig.add_subplot(121)
ax.set_xlabel("radial distance", size=20) 
ax.hist(r_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.axvline(r_scale, color="black", label="core radius")
ax.legend(loc=0) 

ax=fig.add_subplot(122)
ax.set_xlabel("speed", size=20) 
ax.hist(v_agama, bins=30, edgecolor="black", alpha=.5, label ="Agama")

ax.legend(loc=0) 


In [ ]:
pos_0 = np.r_[x[-1], y[-1], z[-1]]
vel_0 = np.r_[vx[-1], vy[-1], vz[-1]]
print(pos_0)
print(vel_0) 

In [ ]:
print("make this a function")

def shift_to_gc(xv, pos_0, vel_0): 

    x = xv[:,0] + pos_0[0]
    y = xv[:,1] + pos_0[1]
    z = xv[:,2] + pos_0[2]

    vx = xv[:,3] + vel_0[0]
    vy = xv[:,4] + vel_0[1]
    vz = xv[:,5] + vel_0[2]


    print("checks")
    print(np.mean(x), np.mean(y), np.mean(z))
    print(pos_0)


    print(np.mean(vx), np.mean(vy), np.mean(vz))
    print(vel_0)

    return np.column_stack((x, y, z)), np.column_stack((vx, vy, vz))


In [ ]:
# pos_0 = np.column_stack((x, y, z))
# vel_0 = np.column_stack((vx, vy, vz))
# vel_0.shape

pos_0 , vel_0 = shift_to_gc(xv, pos_0, vel_0)

In [ ]:
# np.sum(mass/1e5)
# np.savetxt(OUTPUT_DIR / "ngc6569_mass.txt", mass)

In [ ]:
time_unit=u.kpc.to(u.km)*u.s.to(u.Gyr)
time_unit

In [ ]:
tmax = -tfin.to(u.Gyr).value/time_unit
print("maximum time", tmax)
print(t_scipy[-1])

In [ ]:
def get_tau(kmax, time_unit): 
    return 2**(-kmax)*time_unit

In [ ]:
kmax = 13
# tau = 2**(-kmax)*time_unit
tau = get_tau(kmax, time_unit) 
print("time step:", tau) 
nt=int(tmax/tau) + 1
print("number of time steps:", nt)

eps = 1/1000  # II 
eps = .1/1000 # I 
# eps_power = -4
# eps = (2**eps_power)/1000
print("softening length:", eps) 

In [ ]:
(nt)*tau

In [ ]:
print(nt)

In [ ]:
downsample=20
#filename="ngc_6569_runI"
#nt=1000

In [ ]:
pot_use

In [ ]:
t1 = time()
sim_data = kdk_leapfrog(pot_use, pos_0, vel_0, 
                        mass, nt, tau, agama.G, eps, 
                        time_unit, downsample,last_snapshot=False)
t2=time()
print("run time", t2-t1, (t2-t1)/60)

In [ ]:
print(len(sim_data))
sim_data.keys()

In [ ]:
# get mass loss from energy criteria

In [ ]:
bound_data = get_bound_by_energy(sim_data) 
cluster_mass = bound_data["mass"]
time = sim_data["time"]

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
#ax.set_ylim(.9999,1)
ax.plot(time, cluster_mass/cluster_mass[0])
ax.set_xlabel("Time (Myr)")
ax.set_ylabel(r"Bound Mass Fraction, $M/M_0$")

In [ ]:
# Create figure
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)
ax.set_aspect('equal')
ax.set_xlabel('X (kpc)')
ax.set_ylabel('Y (kpc)')
L = 4
ax.set_xlim(-L, L)
ax.set_ylim(-L, L)
ax.plot(orbit[1][:,0], orbit[1][:,1], color="black", alpha=.1, label="Agama")
# gala orbit
#ax.plot(orbit.pos.x, orbit.pos.y, color="black", alpha=.5, label="Gala")
pts, = ax.plot([], [], "o", color="blue", alpha=0.25, markersize=1)
plt.close()

n_snap = sim_data["pos"].shape[0]

# Initialization function
def init():
    pts.set_data([], [])
    return pts,

def draw(i):
    pos = sim_data["pos"][i]
    x = pos[:, 0]
    y = pos[:, 1]
    z = pos[:, 2]

    pts.set_data(x, y)
    time = np.round(sim_data["time"][i], 0)

    ax.set_title(f'NGC6569 5 (X-Y Plane) - Time: {time} Myr')
    return pts,

anim = FuncAnimation(fig, draw, init_func=init, frames=n_snap, interval=50, blit=True)
# anim.save(OUTPUT_DIR / "6569_2d_II.mp4")
HTML(anim.to_jshtml())

In [ ]:
cluster_frame_data = get_cluster_data(bound_data, sim_data)

In [ ]:
# # animation in the cluster frame

# # Create figure
# fig = plt.figure(figsize=(6, 6))
# ax = fig.add_subplot(111)
# ax.set_aspect('equal')
# ax.set_xlabel('X [kpc]', size=20)
# ax.set_ylabel('Y [kpc]', size=20)
# L = .3
# ax.set_xlim(-L, L)
# ax.set_ylim(-L, L)
# pts, = ax.plot([], [], "o", color="blue", alpha=0.1, markersize=1)
# # free, = ax.plot([], [], "o", color = "red", alpha = .3, markersize=1, label="stripped")
# # circ, = ax.plot([], [], color = "orange", alpha =.5 ,label = "Jacobi radius")
# # circ2, = ax.plot([], [], color = "black", alpha=.5, label = "energy based bound radius")
# # ax.legend(loc=2, fontsize="large")
# plt.close()
# theta = np.linspace(0, 2*np.pi, 1000)

# n_snap = cluster_frame_data["pos"].shape[0]

# # Initialization function
# def init():
#     pts.set_data([], [])
#     # circ.set_data([], [])
#     # circ2.set_data([], [])
#     # free.set_data([], [])
#     return pts,  # circ, circ2, free,

# def draw(i):
#     pos = cluster_frame_data["pos"][i]
#     x = pos[:, 0] - np.mean(pos[:, 0])
#     y = pos[:, 1] - np.mean(pos[:, 1])
#     # z = pos[:, 2] - np.mean(pos[:, 2])

#     pts.set_data(x, y)
#     # free_x = x[free_mask]
#     # free_y = y[free_mask]
#     # free.set_data(free_x, free_y)
#     # x = r_tidal[i]*np.cos(theta)
#     # y = r_tidal[i]*np.sin(theta)
#     # circ.set_data(x, y)
#     # r_bound = bound_data["r_bound"][i]

#     # x = r_bound*np.cos(theta)
#     # y = r_bound*np.sin(theta)
#     # circ2.set_data(x, y)
#     time = np.round(cluster_frame_data["time"][i], 0)

#     ax.set_title(f'Cluster Frame - Time: {time} Myr')

#     return pts,  # circ, circ2, free,

# anim = FuncAnimation(fig, draw, init_func=init, frames=n_snap, interval=100, blit=True)
# # anim.save(OUTPUT_DIR / "pal5test_blob_rj.mp4")
# HTML(anim.to_jshtml())

In [ ]:
kvals = [10, 11, 12, 13, 14, 15, 16] 

names = []

for k in kvals: 
    name = "run_kmax_" + str(k) 
    names.append(name) 
    tau = get_tau(k, time_unit)

    # uncomment this
    #sim_data = kdk_leapfrog(pot_use, pos_0, vel_0, 
                        # mass, nt, tau, agama.G, eps, 
                        # time_unit, downsample,last_snapshot=False)

    np.savez_compressed(OUTPUT_DIR / f"{name}.npz", **sim_data)



# then make the mass loss plots for each one. 

    